# Task 1 Optuna screening
Run setup, then the search cell. Existing model outputs are preserved. Eight trials × five epochs are screening evidence, not final models. Run only one search process per output folder.


In [ ]:
import torch
print(torch.__version__)
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
# Before starting a new experiment, fetch the implementation you pushed.
# When resuming, keep the source revision used by your checkpoint.
subprocess.run(['git', 'pull', '--ff-only'], check=True)
assert (repo / 'training/train_task1_spatial16.py').is_file(), 'Push the new files first'
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-optimization.txt'], check=True)


If Colab requests a session restart after installation, restart and rerun setup. Keep the original source revision when resuming; provenance checks reject changed training sources.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/GenAI_A1')
os.chdir(repo)
raw = repo / 'data/raw/oxford_pets'
raw.mkdir(parents=True, exist_ok=True)
for name in ['images', 'annotations']:
    if not (raw / name).is_dir():
        subprocess.run(['wget', '-c', '-P', str(raw), f'https://thor.robots.ox.ac.uk/pets/{name}.tar.gz'], check=True)
        subprocess.run(['tar', '-xzf', str(raw / f'{name}.tar.gz'), '-C', str(raw)], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_pets'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.generate_manifests'], check=True)
output = Path('/content/drive/MyDrive/GenAI_A1/task1-optuna')


## Search
Full training split and all validation cases. Four-hour soft session budget; runtime availability is not guaranteed. Rerun the same cell to resume a paused trial and finish the remaining trials. `--trials 8` means eight total trials, not eight additional trials. Do not change source/config between sessions.


In [ ]:
subprocess.run([sys.executable, '-m', 'optimization.optimize_task1',
    '--device', 'cuda', '--trials', '8', '--screen-epochs', '5',
    '--max-hours', '4', '--output-dir', str(output)], check=True)


In [ ]:
import json
report = json.loads((output / 'leaderboard.json').read_text())
print('Unchanged-input baseline:', report['input_baseline'])
for trial in report['ranked_trials']:
    print(trial['number'], round(trial['score'], 6), trial['params'])


Share `leaderboard.json` after screening. Each trial has MLflow logs, history, and checkpoints. Rank uses fixed-alpha score at the final screening epoch, so use `last.pt` to inspect that candidate, not its training-alpha-selected `best.pt`. `selected_config.yaml` is provisional: review top candidates before equal-budget full training. Preserve the entire task1-optuna folder, including study.json. No official test images are evaluated.
